# Notebook 04 — Model Training & Evaluation

## Purpose
Train both core valuation models for PropertyIQ using engineered features from Notebook 03.
Generate honest evaluation metrics on held-out validation sets.
Compute tree-based confidence scores for every prediction.
Save production-ready models with full metadata registry.

## Inputs
- `data/processed/features_train.csv` — 69,353 rows × 15 cols (sale price training)
- `data/processed/features_val.csv` — 11,558 rows × 15 cols (sale price evaluation)
- `data/processed/features_rent_train.csv` — 2,180 rows × 15 cols (rental training)
- `data/processed/features_rent_drift.csv` — 1,743 rows × 15 cols (rental evaluation)
- Features: city_tier_encoded, bhk, total_sqft, bath, bath_per_bhk, sqft_per_bhk, is_large_property, city_median_price_sqft, locality_median_price_sqft, price_sqft_city_zscore, rbi_hpi_avg, hpi_tier_interaction, sqft_city_interaction, bhk_sqft_ratio

## Outputs
- `models/sale_price_v1.pkl` — Trained RandomForest for ₹/sqft sale prices
- `models/rental_value_v1.pkl` — Trained RandomForest for monthly rental values
- `outputs/model_registry.json` — Complete metadata including hyperparams, metrics, per-city MAPE, feature importances
- `outputs/feature_importance_sale.png` — Feature importance visualization
- `outputs/model_evaluation_plots.png` — 2×2 diagnostic plots (actual vs pred, residuals, error dist, per-city MAPE)

## Model Design Rationale

**Why RandomForestRegressor?**
- Captures non-linear locality effects (correlation=0.938 with locality_median_price_sqft)
- Provides tree-level variance for uncertainty quantification → confidence scores
- Robust to outliers in high-value properties — no scaling required
- OOB score provides free validation estimate during training
- Handles zero-variance features (rbi_hpi_avg) gracefully

**Expected Performance**
- Sale model: 8-14% MAPE (strong locality signal)
- Rental model: 10-18% MAPE (smaller dataset, shorter time window)

In [1]:
# ── CELL 2 ─ Imports & Constants ──────────────

from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json
import logging
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    r2_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Define base paths
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
OUTPUTS_DIR = Path("../outputs")

# Hyperparameters — exact as specified in design
SALE_MODEL_PARAMS = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 4,
    "min_samples_split": 8,
    "max_features": "sqrt",
    "random_state": 42,
    "n_jobs": -1,
    "oob_score": True
}

RENTAL_MODEL_PARAMS = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_leaf": 5,
    "min_samples_split": 10,
    "max_features": "sqrt",
    "random_state": 42,
    "n_jobs": -1,
    "oob_score": True
}

# Feature and target definitions
FEATURE_COLS = [
    "city_tier_encoded",
    "bhk",
    "total_sqft",
    "bath",
    "bath_per_bhk",
    "sqft_per_bhk",
    "is_large_property",
    "city_median_price_sqft",
    "locality_median_price_sqft",
    "price_sqft_city_zscore",
    "rbi_hpi_avg",
    "hpi_tier_interaction",
    "sqft_city_interaction",
    "bhk_sqft_ratio"
]

TARGET_SALE = "price_sqft"
TARGET_RENTAL = "rent_monthly"

# Performance thresholds
MAPE_THRESHOLD = 15.0
CONFIDENCE_THRESHOLD_TRUSTED = 75.0
CONFIDENCE_THRESHOLD_CAUTION = 50.0

logger.info(f"Notebook 04: Model Training & Evaluation initialized")
logger.info(f"Features defined: {len(FEATURE_COLS)} columns")
logger.info(f"Sale model params: {SALE_MODEL_PARAMS['n_estimators']} trees, "
            f"depth={SALE_MODEL_PARAMS['max_depth']}")
logger.info(f"Rental model params: {RENTAL_MODEL_PARAMS['n_estimators']} trees, "
            f"depth={RENTAL_MODEL_PARAMS['max_depth']}")

2026-03-11 12:04:12 | INFO     | Notebook 04: Model Training & Evaluation initialized
2026-03-11 12:04:12 | INFO     | Features defined: 14 columns
2026-03-11 12:04:12 | INFO     | Sale model params: 300 trees, depth=12
2026-03-11 12:04:12 | INFO     | Rental model params: 200 trees, depth=10


In [2]:
# ── CELL 3 ─ Load Feature Files ───────────────
# Load all processed feature CSVs and prepare X/y splits for both models

try:
    logger.info("Loading feature CSV files...")
    
    # Sale price model data
    df_train = pd.read_csv(DATA_PROCESSED / "features_train.csv")
    df_val = pd.read_csv(DATA_PROCESSED / "features_val.csv")
    
    # Rental model data
    df_rent_train = pd.read_csv(DATA_PROCESSED / "features_rent_train.csv")
    df_rent_drift = pd.read_csv(DATA_PROCESSED / "features_rent_drift.csv")
    
    logger.info(f"✓ Loaded features_train.csv: {df_train.shape}")
    logger.info(f"✓ Loaded features_val.csv: {df_val.shape}")
    logger.info(f"✓ Loaded features_rent_train.csv: {df_rent_train.shape}")
    logger.info(f"✓ Loaded features_rent_drift.csv: {df_rent_drift.shape}")
    
    # Extract X and y for sale price model
    X_train = df_train[FEATURE_COLS].copy()
    y_train = df_train[TARGET_SALE].copy()
    
    X_val = df_val[FEATURE_COLS].copy()
    y_val = df_val[TARGET_SALE].copy()
    
    # Extract X and y for rental model
    X_rent_train = df_rent_train[FEATURE_COLS].copy()
    y_rent_train = df_rent_train[TARGET_RENTAL].copy()
    
    X_rent_val = df_rent_drift[FEATURE_COLS].copy()
    y_rent_val = df_rent_drift[TARGET_RENTAL].copy()
    
    # Handle NaN values (rbi_hpi_avg may have zero variance in training)
    nan_count_before = {
        'X_train': X_train.isna().sum().sum(),
        'X_val': X_val.isna().sum().sum(),
        'X_rent_train': X_rent_train.isna().sum().sum(),
        'X_rent_val': X_rent_val.isna().sum().sum()
    }
    
    X_train.fillna(0, inplace=True)
    X_val.fillna(0, inplace=True)
    X_rent_train.fillna(0, inplace=True)
    X_rent_val.fillna(0, inplace=True)
    
    logger.info(f"NaN values filled with 0: {nan_count_before}")
    
    # Print shapes
    print("\n" + "="*70)
    print("FEATURE MATRICES — TRAINING & VALIDATION SETS")
    print("="*70)
    print(f"Sale Price Model:")
    print(f"  X_train: {X_train.shape}    y_train: {y_train.shape}")
    print(f"  X_val:   {X_val.shape}    y_val:   {y_val.shape}")
    print(f"\nRental Value Model:")
    print(f"  X_rent_train: {X_rent_train.shape}    y_rent_train: {y_rent_train.shape}")
    print(f"  X_rent_val:   {X_rent_val.shape}    y_rent_val:   {y_rent_val.shape}")
    print("="*70 + "\n")
    
    # Critical assertions
    assert X_train.shape[1] == 14, f"X_train has {X_train.shape[1]} features, expected 14"
    assert X_val.shape[1] == 14, f"X_val has {X_val.shape[1]} features, expected 14"
    assert X_rent_train.shape[1] == 14, f"X_rent_train has {X_rent_train.shape[1]} features, expected 14"
    assert X_rent_val.shape[1] == 14, f"X_rent_val has {X_rent_val.shape[1]} features, expected 14"
    
    assert X_train.isna().sum().sum() == 0, "NaNs remain in X_train"
    assert X_val.isna().sum().sum() == 0, "NaNs remain in X_val"
    assert X_rent_train.isna().sum().sum() == 0, "NaNs remain in X_rent_train"
    assert X_rent_val.isna().sum().sum() == 0, "NaNs remain in X_rent_val"
    
    assert y_train.isna().sum() == 0, "NaNs in y_train"
    assert y_val.isna().sum() == 0, "NaNs in y_val"
    assert y_rent_train.isna().sum() == 0, "NaNs in y_rent_train"
    assert y_rent_val.isna().sum() == 0, "NaNs in y_rent_val"
    
    logger.info("✓ All assertions passed: features, shapes, NaNs")
    
except Exception as e:
    logger.error(f"Error loading feature files: {e}")
    raise

2026-03-11 12:04:12 | INFO     | Loading feature CSV files...
2026-03-11 12:04:12 | INFO     | ✓ Loaded features_train.csv: (69353, 15)
2026-03-11 12:04:12 | INFO     | ✓ Loaded features_val.csv: (11558, 15)
2026-03-11 12:04:12 | INFO     | ✓ Loaded features_rent_train.csv: (2180, 15)
2026-03-11 12:04:12 | INFO     | ✓ Loaded features_rent_drift.csv: (1743, 15)
2026-03-11 12:04:12 | INFO     | NaN values filled with 0: {'X_train': np.int64(0), 'X_val': np.int64(0), 'X_rent_train': np.int64(0), 'X_rent_val': np.int64(0)}
2026-03-11 12:04:12 | INFO     | ✓ All assertions passed: features, shapes, NaNs



FEATURE MATRICES — TRAINING & VALIDATION SETS
Sale Price Model:
  X_train: (69353, 14)    y_train: (69353,)
  X_val:   (11558, 14)    y_val:   (11558,)

Rental Value Model:
  X_rent_train: (2180, 14)    y_rent_train: (2180,)
  X_rent_val:   (1743, 14)    y_rent_val:   (1743,)



In [3]:
# ── CELL 4 ─ Train Sale Price Model ───────────
# Train RandomForestRegressor on engineered features
# Features: 14 columns | Target: price_sqft | Training rows: 69,353

logger.info("Training Sale Price Model...")
logger.info(f"Training on {len(X_train):,} rows, {X_train.shape[1]} features")
logger.info(f"Hyperparameters: {SALE_MODEL_PARAMS}")

try:
    sale_model = RandomForestRegressor(**SALE_MODEL_PARAMS)
    
    start_time = time.time()
    sale_model.fit(X_train, y_train)
    elapsed = time.time() - start_time
    
    logger.info(f"✓ Training complete in {elapsed:.1f} seconds")
    logger.info(f"✓ OOB R² Score: {sale_model.oob_score_:.4f}")
    
    print("\n" + "="*70)
    print("SALE PRICE MODEL — TRAINING SUMMARY")
    print("="*70)
    print(f"Training time: {elapsed:.1f} seconds")
    print(f"OOB R² Score: {sale_model.oob_score_:.4f}")
    print(f"  (OOB = out-of-bag, free validation estimate during training)")
    print(f"Trees fitted: {sale_model.n_estimators_}")
    print(f"Max depth: {sale_model.max_depth}")
    print("="*70 + "\n")
    
    # Assertions
    assert hasattr(sale_model, 'n_estimators_'), "Model not properly fitted"
    assert sale_model.oob_score_ is not None, "OOB score not computed"
    assert sale_model.oob_score_ > 0, f"OOB score should be positive, got {sale_model.oob_score_}"
    
    logger.info("✓ Sale model training assertions passed")
    
except Exception as e:
    logger.error(f"Error training sale model: {e}")
    raise

2026-03-11 12:04:12 | INFO     | Training Sale Price Model...
2026-03-11 12:04:12 | INFO     | Training on 69,353 rows, 14 features
2026-03-11 12:04:12 | INFO     | Hyperparameters: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 4, 'min_samples_split': 8, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1, 'oob_score': True}
2026-03-11 12:04:15 | INFO     | ✓ Training complete in 3.1 seconds
2026-03-11 12:04:15 | INFO     | ✓ OOB R² Score: 0.9917
2026-03-11 12:04:15 | ERROR    | Error training sale model: 'RandomForestRegressor' object has no attribute 'n_estimators_'



SALE PRICE MODEL — TRAINING SUMMARY
Training time: 3.1 seconds
OOB R² Score: 0.9917
  (OOB = out-of-bag, free validation estimate during training)


AttributeError: 'RandomForestRegressor' object has no attribute 'n_estimators_'

In [ ]:
# ── CELL 5 ─ Evaluate Sale Model ──────────────
# Evaluate on held-out validation set
# Compute MAPE, MAE, R², and per-city breakdown

logger.info("Evaluating Sale Price Model on validation set...")

try:
    # Generate predictions
    y_pred_val = sale_model.predict(X_val)
    
    # Compute overall metrics
    mape = mean_absolute_percentage_error(y_val, y_pred_val) * 100
    mae = mean_absolute_error(y_val, y_pred_val)
    r2 = r2_score(y_val, y_pred_val)
    
    # Per-city MAPE breakdown
    df_eval = df_val.copy()
    df_eval['y_pred'] = y_pred_val
    df_eval['abs_pct_err'] = (
        np.abs(df_eval[TARGET_SALE] - df_eval['y_pred'])
        / df_eval[TARGET_SALE] * 100
    )
    city_mape = df_eval.groupby('city')['abs_pct_err'].mean().sort_values()
    
    logger.info(f"Overall MAPE: {mape:.2f}%")
    logger.info(f"Overall MAE: ₹{mae:,.0f}/sqft")
    logger.info(f"Overall R²: {r2:.4f}")
    
    # Print evaluation results box
    print("\n" + "╔" + "═"*68 + "╗")
    print("║" + " SALE PRICE MODEL — VALIDATION RESULTS".center(68) + "║")
    print("╠" + "═"*68 + "╣")
    print(f"║ Dataset        : features_val.csv".ljust(69) + "║")
    print(f"║ Rows evaluated : {len(X_val):,}".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ Overall MAPE   : {mape:6.2f}%".ljust(69) + "║")
    print(f"║ Overall MAE    : ₹{mae:,.0f}/sqft".ljust(69) + "║")
    print(f"║ R² Score       : {r2:6.4f}".ljust(69) + "║")
    print(f"║ OOB R²         : {sale_model.oob_score_:6.4f}".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print("║ Per-City MAPE (Best → Worst):".ljust(69) + "║")
    for city, city_mape_val in city_mape.items():
        print(f"║   {city:12s} : {city_mape_val:6.2f}%".ljust(69) + "║")
    print("║" + " "*68 + "║")
    
    # Status
    if mape > MAPE_THRESHOLD:
        status = f"⚠ RETRAIN NEEDED (MAPE {mape:.1f}% > {MAPE_THRESHOLD}%)"
    else:
        status = f"✓ PASS (MAPE {mape:.1f}% < {MAPE_THRESHOLD}%)"
    
    print(f"║ Status: {status}".ljust(69) + "║")
    print("╚" + "═"*68 + "╝\n")
    
    # Logging decision
    if mape > MAPE_THRESHOLD:
        logger.warning(
            f"MAPE {mape:.1f}% exceeds threshold {MAPE_THRESHOLD}%. "
            "Consider retraining with more data or refined hyperparameters."
        )
    else:
        logger.info(
            f"MAPE {mape:.1f}% within acceptable threshold {MAPE_THRESHOLD}%. "
            "Model ready for production."
        )
    
    # Assertions
    assert r2 > 0.0, f"R² score should be positive, got {r2}"
    assert mape < 50.0, f"MAPE sanity check failed: {mape:.1f}% seems too high"
    
    logger.info("✓ Sale model evaluation assertions passed")
    
except Exception as e:
    logger.error(f"Error evaluating sale model: {e}")
    raise

In [ ]:
# ── CELL 6 ─ Train Rental Model ───────────────
# Train RandomForestRegressor for monthly rental values
# Features: 14 columns | Target: rent_monthly | Training rows: 2,180

logger.info("Training Rental Value Model...")
logger.info(f"Training on {len(X_rent_train):,} rows, {X_rent_train.shape[1]} features")
logger.info(f"Note: All rental data from Apr-Jul 2022 (89-day window)")
logger.info(f"Hyperparameters: {RENTAL_MODEL_PARAMS}")

try:
    rental_model = RandomForestRegressor(**RENTAL_MODEL_PARAMS)
    
    start_time = time.time()
    rental_model.fit(X_rent_train, y_rent_train)
    elapsed = time.time() - start_time
    
    logger.info(f"✓ Training complete in {elapsed:.1f} seconds")
    logger.info(f"✓ OOB R² Score: {rental_model.oob_score_:.4f}")
    
    print("\n" + "="*70)
    print("RENTAL VALUE MODEL — TRAINING SUMMARY")
    print("="*70)
    print(f"Training time: {elapsed:.1f} seconds")
    print(f"OOB R² Score: {rental_model.oob_score_:.4f}")
    print(f"  (OOB = out-of-bag, free validation estimate during training)")
    print(f"Trees fitted: {rental_model.n_estimators_}")
    print(f"Max depth: {rental_model.max_depth}")
    print("\nNote: All rental data sourced from Apr-Jul 2022 (89-day window)")
    print("="*70 + "\n")
    
    # Assertions
    assert hasattr(rental_model, 'n_estimators_'), "Model not properly fitted"
    assert rental_model.oob_score_ is not None, "OOB score not computed"
    assert rental_model.oob_score_ > 0, f"OOB score should be positive, got {rental_model.oob_score_}"
    
    logger.info("✓ Rental model training assertions passed")
    
except Exception as e:
    logger.error(f"Error training rental model: {e}")
    raise

In [ ]:
# ── CELL 7 ─ Evaluate Rental Model ────────────
# Evaluate on rent_drift (only validation set available for rental data)
# Note: All rental data from single 89-day window Apr-Jul 2022

logger.info("Evaluating Rental Value Model on drift set...")
logger.warning(
    "Note: Rental evaluation uses rent_drift as validation set. "
    "All rental data from Apr-Jul 2022 (89-day window). "
    "No true temporal holdout possible for rental model. "
    "MAPE should be interpreted with this dataset context."
)

try:
    # Generate predictions
    y_rent_pred_val = rental_model.predict(X_rent_val)
    
    # Compute overall metrics
    rent_mape = mean_absolute_percentage_error(y_rent_val, y_rent_pred_val) * 100
    rent_mae = mean_absolute_error(y_rent_val, y_rent_pred_val)
    rent_r2 = r2_score(y_rent_val, y_rent_pred_val)
    
    logger.info(f"Overall MAPE: {rent_mape:.2f}%")
    logger.info(f"Overall MAE: ₹{rent_mae:,.0f}/month")
    logger.info(f"Overall R²: {rent_r2:.4f}")
    
    # Print evaluation results box
    print("\n" + "╔" + "═"*68 + "╗")
    print("║" + " RENTAL VALUE MODEL — VALIDATION RESULTS".center(68) + "║")
    print("╠" + "═"*68 + "╣")
    print(f"║ Dataset        : features_rent_drift.csv".ljust(69) + "║")
    print(f"║ Rows evaluated : {len(X_rent_val):,}".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print(f"║ Overall MAPE   : {rent_mape:6.2f}%".ljust(69) + "║")
    print(f"║ Overall MAE    : ₹{rent_mae:,.0f}/month".ljust(69) + "║")
    print(f"║ R² Score       : {rent_r2:6.4f}".ljust(69) + "║")
    print(f"║ OOB R²         : {rental_model.oob_score_:6.4f}".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print("║ ⚠ Limitation: All data from Apr-Jul 2022 (89-day window)".ljust(69) + "║")
    print("║   No temporal holdout available for rental dataset.".ljust(69) + "║")
    print("║" + " "*68 + "║")
    
    # Status
    if rent_mape > MAPE_THRESHOLD:
        status = f"⚠ REVIEW (MAPE {rent_mape:.1f}% > {MAPE_THRESHOLD}%)"
    else:
        status = f"✓ PASS (MAPE {rent_mape:.1f}% < {MAPE_THRESHOLD}%)"
    
    print(f"║ Status: {status}".ljust(69) + "║")
    print("╚" + "═"*68 + "╝\n")
    
    # Logging decision
    if rent_mape > MAPE_THRESHOLD:
        logger.warning(
            f"Rental MAPE {rent_mape:.1f}% exceeds threshold {MAPE_THRESHOLD}%. "
            "Note: Limited rental data (2,180 samples) may impact model performance."
        )
    else:
        logger.info(
            f"Rental MAPE {rent_mape:.1f}% within acceptable threshold. "
            "Model ready for production with temporal limitations noted."
        )
    
    # Assertions
    assert rent_r2 > 0.0, f"R² score should be positive, got {rent_r2}"
    assert rent_mape < 50.0, f"MAPE sanity check failed: {rent_mape:.1f}% seems too high"
    
    logger.info("✓ Rental model evaluation assertions passed")
    
except Exception as e:
    logger.error(f"Error evaluating rental model: {e}")
    raise

In [ ]:
# ── CELL 8 ─ Confidence Scores ────────────────
# Implement tree-based confidence scoring for production predictions
# Confidence = f(tree_variance, model_error) → 0-100 business score

logger.info("Implementing confidence score functions...")

def compute_confidence(model, X_input, base_mape):
    """
    Computes prediction confidence score 0-100 using tree variance.
    
    Method:
      1. Extract predictions from all trees (n_estimators)
      2. Compute coefficient of variation: CV = std(tree_preds) / mean(tree_preds)
      3. Convert to raw confidence: 1 - CV*5 (clipped 0-1)
      4. Apply MAPE penalty: mape_factor = base_mape / 100
      5. Final confidence = raw_conf * (1 - mape_factor*0.5) * 100
    
    Interpretation:
      >= 75: TRUSTED (teal)   — safe to proceed with loan evaluation
      50-75: CAUTION (amber)  — verify before deciding
      < 50:  DANGER (red)     — field verification required
    
    Args:
        model: Fitted RandomForestRegressor with oob_score=True
        X_input: Feature array (1 row or batch)
        base_mape: Model's validation MAPE (float, percent)
    
    Returns:
        np.array: Confidence scores 0.0-100.0 (one per row in X_input)
    """
    # Get predictions from all individual trees
    tree_preds = np.array([
        tree.predict(X_input)
        for tree in model.estimators_
    ])  # Shape: (n_estimators, n_samples)
    
    # Compute mean and std of tree predictions
    mean_pred = tree_preds.mean(axis=0)
    std_pred = tree_preds.std(axis=0)
    
    # Coefficient of variation (std / mean)
    # Add small epsilon to avoid division by zero
    cv = std_pred / (np.abs(mean_pred) + 1e-9)
    
    # Convert CV to raw confidence (0-1)
    raw_conf = np.clip(1 - cv * 5, 0, 1)
    
    # Apply MAPE penalty (higher model error → lower confidence)
    mape_penalty = base_mape / 100
    final_conf = raw_conf * (1 - mape_penalty * 0.5)
    
    # Scale to 0-100
    return np.clip(final_conf * 100, 0, 100)


def get_trust_signal(confidence):
    """
    Translates confidence score to business decision signal.
    
    This function outputs ONLY plain English business language.
    Rahul (the stakeholder) never sees technical confidence numbers,
    only these trust signals.
    
    Args:
        confidence (float): Confidence score from compute_confidence()
    
    Returns:
        dict with keys:
            level: "trusted" / "caution" / "danger"
            label: Plain English headline
            message: Plain English explanation
            color: "teal" / "amber" / "red"
    """
    if confidence >= CONFIDENCE_THRESHOLD_TRUSTED:
        return {
            "level": "trusted",
            "label": "VALUATION TRUSTED",
            "message": (
                "Market data is current and consistent. "
                "Safe to proceed with loan evaluation."
            ),
            "color": "teal"
        }
    elif confidence >= CONFIDENCE_THRESHOLD_CAUTION:
        return {
            "level": "caution",
            "label": "VERIFY BEFORE DECIDING",
            "message": (
                "Market conditions have shifted recently. "
                "Estimate may be 5-10% off. "
                "Recommend field verification."
            ),
            "color": "amber"
        }
    else:
        return {
            "level": "danger",
            "label": "FIELD VERIFICATION REQUIRED",
            "message": (
                "Significant market movement detected. "
                "Estimate could be 15-20% below current market. "
                "Do not use without independent verification."
            ),
            "color": "red"
        }


logger.info("✓ Confidence functions defined")

# Test on 5 sample properties to demonstrate confidence scoring
print("\n" + "="*75)
print("SAMPLE PREDICTIONS WITH CONFIDENCE SCORES")
print("="*75)

sample_indices = [0, 1000, 5000, 10000, 20000]
X_sample = X_val.iloc[sample_indices].reset_index(drop=True)
y_sample = y_val.iloc[sample_indices].reset_index(drop=True)

for i in range(len(X_sample)):
    # Get prediction for this property
    X_row = X_sample.iloc[i:i+1].values
    y_actual = y_sample.iloc[i]
    
    pred = sale_model.predict(X_row)[0]
    confidence_scores = compute_confidence(sale_model, X_row, mape)
    conf = confidence_scores[0]
    
    signal = get_trust_signal(conf)
    err_pct = abs(pred - y_actual) / y_actual * 100
    
    print(f"\nProperty {i+1}:")
    print(f"  Actual Price:   ₹{y_actual:>9,.0f}/sqft")
    print(f"  Predicted:      ₹{pred:>9,.0f}/sqft")
    print(f"  Error:          {err_pct:>9.1f}%")
    print(f"  Confidence:     {conf:>9.1f}%")
    print(f"  Trust Signal:   {signal['label']}")

print("\n" + "="*75 + "\n")

logger.info("✓ Confidence score calculation and trust signal translation working")

In [ ]:
# ── CELL 9 ─ Feature Importance ───────────────
# Analyze which features drive predictions in the sale price model

logger.info("Computing feature importances...")

try:
    # Extract feature importances
    importances = pd.Series(
        sale_model.feature_importances_,
        index=FEATURE_COLS
    ).sort_values(ascending=False)
    
    logger.info(f"✓ Top feature: {importances.index[0]} ({importances.iloc[0]:.4f})")
    
    # Print importance table
    print("\n" + "="*70)
    print("FEATURE IMPORTANCE — SALE PRICE MODEL")
    print("="*70)
    
    max_importance = importances.max()
    for i, (feature, importance) in enumerate(importances.items(), 1):
        bar_length = int((importance / max_importance) * 50)
        bar = "█" * bar_length + "░" * (50 - bar_length)
        print(f"{i:2d}. {feature:30s} : {importance:7.4f}  {bar}")
    
    print("="*70 + "\n")
    
    # Assertion: top feature must be location-based
    location_features = {
        'locality_median_price_sqft',
        'city_median_price_sqft',
        'price_sqft_city_zscore'
    }
    top_feature = importances.index[0]
    assert top_feature in location_features, (
        f"Top feature '{top_feature}' is not a location-based feature. "
        f"Expected one of {location_features}"
    )
    
    logger.info(f"✓ Top feature assertion passed: {top_feature} is location-based")
    
    # Create feature importance visualization
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = ['#1f77b4' if i < 5 else '#808080' for i in range(len(importances))]
    
    importances.sort_values().plot(
        kind='barh',
        ax=ax,
        color=colors[::-1]
    )
    
    ax.set_xlabel('Importance Score', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
    ax.set_title('Feature Importance — Sale Price Model', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    plot_path = OUTPUTS_DIR / "feature_importance_sale.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    logger.info(f"✓ Feature importance plot saved: {plot_path}")
    
    plt.close()
    
except Exception as e:
    logger.error(f"Error analyzing feature importance: {e}")
    raise

In [ ]:
# ── CELL 10 ─ Evaluation Plots ─────────────────
# Create 2×2 diagnostic plots: actual vs pred, residuals, error dist, per-city MAPE

logger.info("Creating evaluation visualization...")

try:
    # Compute residuals and errors
    residuals = y_val - y_pred_val
    pct_errors = np.abs(residuals) / y_val * 100
    
    # Create 2×2 figure with gridspec
    fig = plt.figure(figsize=(14, 10))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)
    
    # ──── PLOT 1: Actual vs Predicted ────
    ax1 = fig.add_subplot(gs[0, 0])
    
    # Get cities for color coding
    cities_val = df_val['city'].values
    city_colors = {
        'Mumbai': '#FF6B6B', 'Delhi': '#4ECDC4', 'Bengaluru': '#45B7D1',
        'Pune': '#FFA07A', 'Hyderabad': '#98D8C8', 'Chennai': '#F7DC6F',
        'Kolkata': '#BB8FCE'
    }
    
    for city in city_colors:
        mask = cities_val == city
        if mask.any():
            ax1.scatter(
                y_val[mask], y_pred_val[mask],
                label=city, alpha=0.6, s=20,
                color=city_colors[city]
            )
    
    # Perfect prediction line
    min_val = min(y_val.min(), y_pred_val.min())
    max_val = max(y_val.max(), y_pred_val.max())
    ax1.plot([min_val, max_val], [min_val, max_val],
             'r--', linewidth=2, label='Perfect Prediction', zorder=10)
    
    ax1.set_xlabel('Actual ₹/sqft', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Predicted ₹/sqft', fontsize=11, fontweight='bold')
    ax1.set_title('Sale Price: Actual vs Predicted', fontsize=12, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # ──── PLOT 2: Residuals vs Predicted ────
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(y_pred_val, residuals, alpha=0.6, s=20, color='#3498db')
    ax2.axhline(y=0, color='r', linestyle='--', linewidth=2, zorder=10)
    ax2.set_xlabel('Predicted ₹/sqft', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Residual ₹/sqft', fontsize=11, fontweight='bold')
    ax2.set_title('Residuals vs Predicted', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # ──── PLOT 3: Error Distribution ────
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.hist(pct_errors, bins=50, color='#2ecc71', alpha=0.7, edgecolor='black')
    ax3.axvline(mape, color='r', linestyle='--', linewidth=2.5,
                label=f'Mean MAPE: {mape:.1f}%', zorder=10)
    ax3.set_xlabel('Percentage Error %', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax3.set_title(f'Error Distribution (MAPE={mape:.1f}%)', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # ──── PLOT 4: Per-City MAPE ────
    ax4 = fig.add_subplot(gs[1, 1])
    
    colors_mape = ['#27ae60' if x <= MAPE_THRESHOLD else '#e74c3c'
                   for x in city_mape.values]
    
    city_mape.sort_values(ascending=True).plot(
        kind='barh',
        ax=ax4,
        color=colors_mape[::-1]
    )
    
    ax4.axvline(MAPE_THRESHOLD, color='orange', linestyle='--', linewidth=2.5,
                label=f'Threshold: {MAPE_THRESHOLD}%', zorder=10)
    ax4.set_xlabel('MAPE %', fontsize=11, fontweight='bold')
    ax4.set_title('Per-City MAPE', fontsize=12, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3, axis='x')
    
    plt.suptitle('Model Evaluation Diagnostics — Sale Price', 
                 fontsize=14, fontweight='bold', y=0.995)
    
    # Save plots
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    plot_path = OUTPUTS_DIR / "model_evaluation_plots.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    logger.info(f"✓ Evaluation plots saved: {plot_path}")
    
    plt.close()
    
except Exception as e:
    logger.error(f"Error creating evaluation plots: {e}")
    raise

In [ ]:
# ── CELL 11 ─ Save Models & Registry ────────────
# Persist trained models and comprehensive metadata for Notebook 05 (drift detection)

logger.info("Saving models and metadata registry...")

try:
    # Create models directory
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    
    # Save trained models using joblib
    sale_model_path = MODELS_DIR / "sale_price_v1.pkl"
    rental_model_path = MODELS_DIR / "rental_value_v1.pkl"
    
    joblib.dump(sale_model, sale_model_path)
    logger.info(f"✓ Sale model saved: {sale_model_path}")
    
    joblib.dump(rental_model, rental_model_path)
    logger.info(f"✓ Rental model saved: {rental_model_path}")
    
    # Build comprehensive model registry
    model_registry = {
        "sale_price_v1": {
            "version": "v1",
            "algorithm": "RandomForestRegressor",
            "trained_on": "features_train.csv",
            "train_rows": int(len(X_train)),
            "val_rows": int(len(X_val)),
            "feature_count": 14,
            "features": FEATURE_COLS,
            "hyperparams": SALE_MODEL_PARAMS,
            "val_mape_percent": round(float(mape), 2),
            "val_mae_per_sqft": round(float(mae), 2),
            "val_r2": round(float(r2), 4),
            "oob_r2": round(float(sale_model.oob_score_), 4),
            "city_mape_percent": city_mape.round(2).to_dict(),
            "mape_threshold_percent": MAPE_THRESHOLD,
            "status": "production",
            "trained_date": str(pd.Timestamp.now().date()),
            "top_feature": importances.index[0],
            "top_feature_importance": round(float(importances.iloc[0]), 4),
            "feature_importances": importances.round(4).to_dict(),
            "confidence_threshold_trusted": CONFIDENCE_THRESHOLD_TRUSTED,
            "confidence_threshold_caution": CONFIDENCE_THRESHOLD_CAUTION
        },
        "rental_value_v1": {
            "version": "v1",
            "algorithm": "RandomForestRegressor",
            "trained_on": "features_rent_train.csv",
            "train_rows": int(len(X_rent_train)),
            "val_rows": int(len(X_rent_val)),
            "feature_count": 14,
            "features": FEATURE_COLS,
            "hyperparams": RENTAL_MODEL_PARAMS,
            "val_mape_percent": round(float(rent_mape), 2),
            "val_mae_per_month": round(float(rent_mae), 2),
            "val_r2": round(float(rent_r2), 4),
            "oob_r2": round(float(rental_model.oob_score_), 4),
            "mape_threshold_percent": MAPE_THRESHOLD,
            "status": "production",
            "trained_date": str(pd.Timestamp.now().date()),
            "confidence_threshold_trusted": CONFIDENCE_THRESHOLD_TRUSTED,
            "confidence_threshold_caution": CONFIDENCE_THRESHOLD_CAUTION,
            "limitation": (
                "Validation set = rent_drift. All rental data from Apr-Jul 2022 "
                "(89-day window). No true temporal holdout possible due to data "
                "collection constraints. MAPE should be interpreted with this context."
            )
        }
    }
    
    # Save registry
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    registry_path = OUTPUTS_DIR / "model_registry.json"
    
    with open(registry_path, 'w') as f:
        json.dump(model_registry, f, indent=2)
    
    logger.info(f"✓ Model registry saved: {registry_path}")
    
    # Print final summary
    print("\n" + "╔" + "═"*68 + "╗")
    print("║" + " NOTEBOOK 04 — FINAL SUMMARY".center(68) + "║")
    print("╠" + "═"*68 + "╣")
    print("║" + " "*68 + "║")
    print("║ SALE PRICE MODEL (sale_price_v1.pkl)".ljust(69) + "║")
    print(f"║   Training rows   : {len(X_train):>10,}".ljust(69) + "║")
    print(f"║   Validation MAPE : {mape:>10.2f}%".ljust(69) + "║")
    print(f"║   Validation R²   : {r2:>10.4f}".ljust(69) + "║")
    print(f"║   OOB R²          : {sale_model.oob_score_:>10.4f}".ljust(69) + "║")
    print(f"║   Status          : PRODUCTION".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print("║ RENTAL MODEL (rental_value_v1.pkl)".ljust(69) + "║")
    print(f"║   Training rows   : {len(X_rent_train):>10,}".ljust(69) + "║")
    print(f"║   Validation MAPE : {rent_mape:>10.2f}%".ljust(69) + "║")
    print(f"║   Validation R²   : {rent_r2:>10.4f}".ljust(69) + "║")
    print(f"║   OOB R²          : {rental_model.oob_score_:>10.4f}".ljust(69) + "║")
    print(f"║   Status          : PRODUCTION".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print("║ FILES SAVED:".ljust(69) + "║")
    print("║   ✓ models/sale_price_v1.pkl".ljust(69) + "║")
    print("║   ✓ models/rental_value_v1.pkl".ljust(69) + "║")
    print("║   ✓ outputs/model_registry.json".ljust(69) + "║")
    print("║   ✓ outputs/feature_importance_sale.png".ljust(69) + "║")
    print("║   ✓ outputs/model_evaluation_plots.png".ljust(69) + "║")
    print("║" + " "*68 + "║")
    print("║ READY FOR NOTEBOOK 05 — DRIFT DETECTION".ljust(69) + "║")
    print("╚" + "═"*68 + "╝\n")
    
    logger.info("="*70)
    logger.info("NOTEBOOK 04 COMPLETE — All models, metrics, and visualizations saved")
    logger.info(f"Sale model MAPE: {mape:.2f}% (target: 8-14%)")
    logger.info(f"Rental model MAPE: {rent_mape:.2f}% (target: 10-18%)")
    logger.info("Ready for drift detection in Notebook 05")
    logger.info("="*70)
    
except Exception as e:
    logger.error(f"Error saving models and registry: {e}")
    raise